# 🚗 Used Car Price Prediction — End-to-End Regression Project

**Dataset:** cars.com used car listings (4,009 rows, 12 columns)

**Pipeline:**
1. Load & inspect data
2. Check for duplicates → remove them
3. Check for missing values → clean & fill them
4. Clean messy text columns (price, mileage, engine)
5. Feature engineering & selection
6. Encode categorical features
7. Train/test split
8. Train multiple models with cross-validation
9. Hyperparameter tuning on the best model
10. Final evaluation + feature importance

Every cell prints out **what it checked** before deciding what cleaning step to apply — nothing is done blindly.


## 1. Setup — install & import libraries

In [ ]:
# Colab already has most of these, but this makes sure versions are compatible
!pip install -q scikit-learn xgboost pandas numpy matplotlib seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
RANDOM_STATE = 42


## 2. Load the dataset

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload used_cars.csv when prompted


In [ ]:
df = pd.read_csv("used_cars.csv")
print("Shape:", df.shape)
df.head()


## 3. Initial inspection

Before cleaning anything, let's actually **check** what we're dealing with: data types, missing values, duplicates, and cardinality of each column. This tells us what needs fixing.

In [ ]:
print("---- INFO ----")
df.info()


In [ ]:
print("---- MISSING VALUES ----")
missing = df.isna().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
print(missing_report[missing_report["missing_count"] > 0])


In [ ]:
print("---- DUPLICATE ROWS CHECK ----")
n_dupes = df.duplicated().sum()
print(f"Number of fully duplicated rows: {n_dupes}")


In [ ]:
print("---- CARDINALITY (unique values per column) ----")
print(df.nunique())


In [ ]:
print("---- SAMPLE OF MESSY COLUMNS ----")
print("price sample:", df["price"].head(3).tolist())
print("milage sample:", df["milage"].head(3).tolist())
print("engine sample:", df["engine"].head(3).tolist())


**What we found:**
- `price` and `milage` are stored as **strings** with `$`, commas, and `mi.` — need numeric conversion.
- `engine` is a free-text description (e.g. "300.0HP 3.7L V6 Cylinder Engine Flex Fuel") — we can extract horsepower and engine size (litres) as numeric features.
- `fuel_type`, `accident`, and `clean_title` have missing values.
- We'll check duplicates again *after* cleaning, since some duplicates only reveal themselves once messy text is normalized.


## 4. Data cleaning — fix messy text columns

In [ ]:
# Work on a copy so we always have the raw df to compare against
clean = df.copy()

# --- price: "$10,300" -> 10300.0 ---
clean["price"] = clean["price"].replace(r"[\$,]", "", regex=True).astype(float)

# --- milage: "51,000 mi." -> 51000.0 ---
clean["milage"] = clean["milage"].replace(r"[a-zA-Z.,]", "", regex=True).str.strip().astype(float)

print("price after cleaning:", clean["price"].head(3).tolist())
print("milage after cleaning:", clean["milage"].head(3).tolist())


In [ ]:
# --- engine: extract horsepower (HP) and engine size (Litres) with regex ---
def extract_hp(text):
    match = re.search(r"(\d+\.?\d*)\s*HP", str(text))
    return float(match.group(1)) if match else np.nan

def extract_liters(text):
    match = re.search(r"(\d+\.?\d*)\s*L", str(text))
    return float(match.group(1)) if match else np.nan

def extract_cylinders(text):
    match = re.search(r"(\d+)\s*Cylinder", str(text))
    return int(match.group(1)) if match else np.nan

clean["horsepower"] = clean["engine"].apply(extract_hp)
clean["engine_liters"] = clean["engine"].apply(extract_liters)
clean["cylinders"] = clean["engine"].apply(extract_cylinders)

print("Extracted from engine text — missing counts:")
print(clean[["horsepower", "engine_liters", "cylinders"]].isna().sum())
clean[["engine", "horsepower", "engine_liters", "cylinders"]].head()


Engine text doesn't always mention HP or cylinder count (e.g. electric vehicles), so some NaNs here are *expected*, not errors. We'll fill them properly in the missing-value step below.

## 5. Duplicate removal (re-checked after cleaning)

In [ ]:
before = clean.shape[0]
dupes_after_cleaning = clean.duplicated().sum()
print(f"Duplicate rows found after cleaning: {dupes_after_cleaning}")

clean = clean.drop_duplicates().reset_index(drop=True)
after = clean.shape[0]
print(f"Rows before: {before} | Rows after removing duplicates: {after} | Removed: {before - after}")


## 6. Handling missing values

We check each column's missing pattern individually rather than blanket-filling everything the same way — categorical vs numeric columns need different strategies.

In [ ]:
print("Missing values remaining:")
print(clean.isna().sum()[clean.isna().sum() > 0])


In [ ]:
# fuel_type: fill with mode (most common fuel type), since it's categorical with few missing
print("fuel_type value counts:")
print(clean["fuel_type"].value_counts())
clean["fuel_type"] = clean["fuel_type"].fillna(clean["fuel_type"].mode()[0])


In [ ]:
# accident: missing likely means "no record reported" -> treat as its own category
print("accident value counts before fill:")
print(clean["accident"].value_counts(dropna=False))
clean["accident"] = clean["accident"].fillna("None reported")


In [ ]:
# clean_title: missing likely means title status wasn't confirmed clean -> its own category
print("clean_title value counts before fill:")
print(clean["clean_title"].value_counts(dropna=False))
clean["clean_title"] = clean["clean_title"].fillna("Unknown")


In [ ]:
# horsepower / engine_liters / cylinders: numeric, fill with median grouped by brand
# (a Toyota Corolla's typical HP is a better guess than the overall dataset median)
for col in ["horsepower", "engine_liters", "cylinders"]:
    clean[col] = clean.groupby("brand")[col].transform(lambda x: x.fillna(x.median()))
    # fallback: if a whole brand group was NaN, fill remaining with global median
    clean[col] = clean[col].fillna(clean[col].median())

print("Missing values after all fills:")
print(clean.isna().sum().sum(), "total missing values remaining")


## 7. Outlier check on the target (price)

In [ ]:
print(clean["price"].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(clean["price"], bins=50, ax=axes[0])
axes[0].set_title("Price distribution")
sns.boxplot(x=clean["price"], ax=axes[1])
axes[1].set_title("Price boxplot")
plt.tight_layout()
plt.show()


In [ ]:
# Price is heavily right-skewed with a few extreme luxury/exotic outliers.
# We cap at the 1st and 99th percentile rather than dropping rows, to keep the dataset size.
low, high = clean["price"].quantile([0.01, 0.99])
print(f"Capping price between {low:.0f} and {high:.0f}")
before_n = clean.shape[0]
clean = clean[(clean["price"] >= low) & (clean["price"] <= high)].reset_index(drop=True)
print(f"Rows before: {before_n} | after outlier trim: {clean.shape[0]}")


## 8. Feature engineering

In [ ]:
# car_age is more informative for pricing than raw model_year
clean["car_age"] = 2026 - clean["model_year"]

# mileage per year of ownership — captures usage intensity
clean["mileage_per_year"] = clean["milage"] / clean["car_age"].replace(0, 1)

# accident/title as binary flags (simpler signal for models than free text)
clean["has_accident"] = clean["accident"].apply(lambda x: 0 if "None" in str(x) else 1)
clean["is_clean_title"] = clean["clean_title"].apply(lambda x: 1 if x == "Yes" else 0)

clean[["model_year", "car_age", "milage", "mileage_per_year", "has_accident", "is_clean_title"]].head()


In [ ]:
# High-cardinality columns (model, engine text, exact colors) add noise without enough
# signal per category. Check cardinality again before deciding what to drop/group.
print("Cardinality check:")
for c in ["brand", "model", "ext_col", "int_col", "transmission", "fuel_type"]:
    print(f"{c}: {clean[c].nunique()} unique values")


In [ ]:
# 'model' has ~1800 unique values across 4000 rows -> too sparse to one-hot encode directly.
# Group rare brands (fewer than 30 listings) into 'Other' to reduce noise/overfitting risk.
brand_counts = clean["brand"].value_counts()
rare_brands = brand_counts[brand_counts < 30].index
clean["brand_grouped"] = clean["brand"].apply(lambda x: "Other" if x in rare_brands else x)
print(f"Brands reduced from {clean['brand'].nunique()} to {clean['brand_grouped'].nunique()} categories")

# Same treatment for exterior/interior color (long tail of rare custom colors)
for col in ["ext_col", "int_col"]:
    counts = clean[col].value_counts()
    rare = counts[counts < 30].index
    clean[col + "_grouped"] = clean[col].apply(lambda x: "Other" if x in rare else x)


## 9. Feature selection

In [ ]:
# Drop columns that are now redundant (raw text) or leak/duplicate info we've already
# extracted into cleaner numeric/categorical features.
drop_cols = ["model", "engine", "accident", "clean_title", "brand", "ext_col", "int_col", "model_year"]
model_df = clean.drop(columns=drop_cols)

print("Final feature set:")
print(model_df.columns.tolist())
model_df.head()


In [ ]:
# Quick correlation check among numeric features (sanity check, not a hard filter)
numeric_cols = model_df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(8, 6))
sns.heatmap(model_df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix (numeric features)")
plt.show()


## 10. Train/test split

In [ ]:
X = model_df.drop(columns=["price"])
y = model_df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train size: {X_train.shape} | Test size: {X_test.shape}")

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)


## 11. Preprocessing pipeline (encoding + scaling)

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols)
])


## 12. Train multiple models with cross-validation

We compare several regressors using 5-fold cross-validation on the training set, scored by RMSE and R². This is how we pick the "best" model before touching the test set.

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Lasso Regression": Lasso(alpha=1.0, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0),
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []

for name, model in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])

    rmse_scores = -cross_val_score(pipe, X_train, y_train, cv=cv,
                                    scoring="neg_root_mean_squared_error", n_jobs=-1)
    r2_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="r2", n_jobs=-1)

    results.append({
        "Model": name,
        "CV_RMSE_mean": rmse_scores.mean(),
        "CV_RMSE_std": rmse_scores.std(),
        "CV_R2_mean": r2_scores.mean(),
        "CV_R2_std": r2_scores.std(),
    })
    print(f"{name:20s} | RMSE: {rmse_scores.mean():,.0f} (+/- {rmse_scores.std():,.0f}) | R2: {r2_scores.mean():.4f}")

results_df = pd.DataFrame(results).sort_values("CV_RMSE_mean")
results_df


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(data=results_df, x="CV_RMSE_mean", y="Model", palette="viridis")
plt.title("Model comparison — Cross-Validated RMSE (lower is better)")
plt.xlabel("RMSE")
plt.show()


## 13. Hyperparameter tuning on the best model

Whichever model scored best in cross-validation gets tuned further with `GridSearchCV`. This code auto-picks the winner from the comparison table above, so it stays correct even if the ranking changes with different data.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
print(f"Best model from CV comparison: {best_model_name}")

param_grids = {
    "Random Forest": {
        "model__n_estimators": [200, 400],
        "model__max_depth": [10, 20, None],
        "model__min_samples_leaf": [1, 2, 4],
    },
    "Gradient Boosting": {
        "model__n_estimators": [200, 400],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [3, 5],
    },
    "XGBoost": {
        "model__n_estimators": [300, 500],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [4, 6],
    },
    "Ridge Regression": {"model__alpha": [0.1, 1.0, 10.0, 50.0]},
    "Lasso Regression": {"model__alpha": [0.01, 0.1, 1.0, 10.0]},
    "Linear Regression": {},
}

best_pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", models[best_model_name])])
grid = param_grids.get(best_model_name, {})

if grid:
    search = GridSearchCV(best_pipe, grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1, verbose=1)
    search.fit(X_train, y_train)
    final_model = search.best_estimator_
    print("Best params:", search.best_params_)
    print(f"Best CV RMSE: {-search.best_score_:,.0f}")
else:
    best_pipe.fit(X_train, y_train)
    final_model = best_pipe
    print("No hyperparameters to tune for this model — fitted as-is.")


## 14. Final evaluation on the held-out test set

In [ ]:
y_pred = final_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test RMSE: {rmse:,.0f}")
print(f"Test MAE:  {mae:,.0f}")
print(f"Test R2:   {r2:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred, alpha=0.4)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")
axes[0].set_title("Predicted vs Actual")

# Residuals
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.4)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("Predicted Price")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residual plot")

plt.tight_layout()
plt.show()


## 15. Feature importance (if supported by the final model)

In [ ]:
model_step = final_model.named_steps["model"]

if hasattr(model_step, "feature_importances_"):
    feature_names = final_model.named_steps["preprocessor"].get_feature_names_out()
    importances = pd.Series(model_step.feature_importances_, index=feature_names)
    top_features = importances.sort_values(ascending=False).head(15)

    plt.figure(figsize=(9, 6))
    sns.barplot(x=top_features.values, y=top_features.index, palette="magma")
    plt.title("Top 15 Feature Importances")
    plt.xlabel("Importance")
    plt.show()
else:
    print(f"{best_model_name} doesn't expose feature_importances_ (e.g. linear models use coefficients instead).")


## 16. Save the final model

In [ ]:
import joblib
joblib.dump(final_model, "used_car_price_model.pkl")
print("Model saved as used_car_price_model.pkl")

from google.colab import files
files.download("used_car_price_model.pkl")


## Summary

- Cleaned `price`, `milage`, and free-text `engine` fields into usable numeric columns.
- Removed duplicate rows (checked both before and after cleaning).
- Filled missing values with column-appropriate strategies (mode for categoricals, brand-grouped median for numerics).
- Engineered `car_age`, `mileage_per_year`, and accident/title flags.
- Grouped rare/high-cardinality categories to reduce noise.
- Compared 6 regression models with 5-fold cross-validation, then tuned the best one with `GridSearchCV`.
- Evaluated on a held-out test set with RMSE, MAE, R², plus predicted-vs-actual and residual plots.

**Next steps to push the score further:** try target/mean encoding for `model` and `brand`, log-transform the price target before training, or stack multiple models.
